# 3. JSON RAG

Flatten nested JSON into path-annotated facts so structure is not lost.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

JSON is hierarchical: the meaning of `payload_kg: 150` depends on the path `products > Carrier-1`. If you flatten naively, or embed the raw JSON string, that context is lost and the retriever cannot tell which product a number belongs to. Path-annotated leaves keep the structure inside the text.

## How it works

1. Load the JSON
2. Walk the tree recursively
3. Emit one fact per leaf with its full key path
4. Use list items' `name` field as the path label so `products[0]` becomes `products > Carrier-1`
5. Embed the facts and retrieve

```text
 {"products":[{"name":"Carrier-1","payload_kg":150}]}
      -> "products > Carrier-1 > payload_kg: 150"
      -> embed -> store -> retrieve -> LLM
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

In [1]:
import sys
sys.path.insert(0, "..")
from common import *

data = json.loads((DATA / "company.json").read_text())

Walk the tree and emit one fact per leaf. The key *path* (`products > Carrier-1 > payload_kg`) preserves the hierarchy.

In [2]:
def flatten(node, path=()):
    if isinstance(node, dict):
        for key, value in node.items():
            yield from flatten(value, path + (key,))
    elif isinstance(node, list):
        for i, item in enumerate(node):
            label = item.get("name", i) if isinstance(item, dict) else i
            yield from flatten(item, path + (str(label),))
    else:
        yield f"{' > '.join(path)}: {node}"

facts = list(flatten(data))
print(len(facts), "facts")
print("\n".join(facts[:8]))

26 facts
company: Nimbus Robotics
founded: 2015
headquarters > city: Pune
headquarters > country: India
mission: Build affordable warehouse robots that work safely beside people.
leadership > Asha Verma > name: Asha Verma
leadership > Asha Verma > role: CEO
leadership > Asha Verma > joined: 2015


In [3]:
store = build_store(facts)
for q in ["How heavy a load can Carrier-1 carry?", "How long is the warranty?", "Who is the CTO?"]:
    print("Q:", q, "\nA:", rag(q, store, k=4)[0], "\n")

W1028 11:46:09.670000 12156 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Q: How heavy a load can Carrier-1 carry? 
A: According to the context, [1] states that Carrier-1 has a payload of 150 kg. Therefore, Carrier-1 can carry a load of 150 kg. 



Q: How long is the warranty? 
A: According to context [1], the warranty length is 2 years. 



Q: Who is the CTO? 
A: According to context [1], the CTO is Daniel Okoye. 



## Alternative strategy: one document per object
Per-leaf facts give precise matches but split related fields apart. Per-object chunks keep all the fields of one product together, which is better for questions that compare several attributes.

In [4]:
object_docs = []
for p in data["products"]:
    object_docs.append(f"Product {p['name']}: " + ", ".join(f"{k}={v}" for k, v in p.items() if k != "name"))
for d in object_docs: print(d)

obj_store = build_store(object_docs)
q = "Which product is cheaper and which lasts longer on a charge, Carrier-1 or Picker-2?"
print("\nper-object answer:", rag(q, obj_store, k=2)[0])
print("\nper-leaf answer  :", rag(q, store, k=6)[0])

Product Carrier-1: type=autonomous mobile robot, payload_kg=150, battery_hours=10, price_usd=18000
Product Picker-2: type=robotic arm picker, payload_kg=8, battery_hours=12, price_usd=26000



per-object answer: Based on the context:

* The Price of Product Picker-2 is $26,000.
* The Price of Product Carrier-1 is $18,000.

Therefore, Product Carrier-1 is cheaper.

As for the battery life, according to the context:

* Product Picker-2 has a battery life of 12 hours.
* Product Carrier-1 has a battery life of 10 hours.

Since 12 hours is greater than 10 hours, Product Picker-2 lasts longer on a charge.



per-leaf answer  : Based on the provided context:

* [5] products > Carrier-1 > name: Carrier-1
* [6] products > Carrier-1 > price_usd: 18000

This means that Carrier-1 is cheaper.

As for which lasts longer on a charge, we need to compare the battery hours of both products. However, there is no information about the battery life of Carrier-1 in the provided context. 

But we do have this information:
* [3] products > Picker-2 > battery_hours: 12

This means that Picker-2 lasts for 12 hours on a charge.

Therefore, I don't know which one lasts longer on a charge without more information about Carrier-1's battery life.


## Nested questions need the path
The warranty and support hours live in different branches. Watch how path text lets retrieval find them.

In [5]:
for q in ["What support channels are available?", "In which city is the company headquartered?"]:
    hits = store.search(q, 3)
    print(q); show(hits, width=80); print("->", answer(q, [h["text"] for h in hits]), "\n")

What support channels are available?
0.613  policies > support > channels > 2: chat...
0.549  policies > support > channels > 1: email...
0.522  policies > support > channels > 0: phone...


-> According to the context, there are three support channels available:

1. Phone (channel 0)
2. Email (channel 1)
3. Chat (channel 2) 

In which city is the company headquartered?
0.589  headquarters > country: India...
0.488  headquarters > city: Pune...
0.417  founded: 2015...


-> According to the context, [2] headquarters > city: Pune, so the company is headquartered in Pune. 



## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Preserves hierarchy via key paths | Many tiny chunks for large documents |
| Works for any JSON shape | Related fields can be separated (mitigate with per-object chunks) |
| No schema knowledge needed | Arrays of scalars lose meaning without a good label |

## When to use it

API responses, configuration files, product catalogues and logs where the structure carries meaning.

## Experiments to try

- Add the leader's name into each product path.
- Flatten with `jq`-style paths (`products[0].name`) and compare retrieval quality.
- Chunk by top-level key instead and measure the hit rate.

## Key takeaways

- Never embed raw JSON strings.
- Keep the key path in the text.
- Choose granularity (leaf vs object) by the kind of question.

---
**Next:** [4. Reliable RAG](../04_reliable_rag/reliable_rag.ipynb)